In [0]:
-- ============================================================
-- Szenario C: Late-arriving Order -- eine Bestellung mit einem
-- Bestelldatum aus der Vergangenheit (VOR Sofias Tier-Upgrade)
-- trifft erst jetzt in der Quelle ein. Nachweis: Der Point-in-Time-
-- Join muss ihr trotzdem die HISTORISCHE (Bronze) Kundenversion
-- zuordnen, nicht die aktuelle (Silver) -- weil order_date zählt,
-- nicht das Ladedatum.
-- ============================================================

-- Late-arriving Bestellung für Sofia (customer_id = 1), datiert auf
-- den 10.01.2024 -- lange vor ihrem Upgrade in Szenario A
INSERT INTO brewflow_oltp.orders VALUES
(202, 1, 1, DATE'2024-01-10', 'in-store', 2.50);

INSERT INTO brewflow_oltp.order_items VALUES
(2002, 202, 1, 1, 2.50);  -- 1x Espresso

-- ============================================================
-- JETZT: Pipeline erneut ausführen! (03 -> 04 -> 05 bzw. Lakeflow Job)
-- ============================================================

-- Validieren: fact_order_items sollte jetzt 34 Zeilen haben
SELECT COUNT(*) AS fact_rows FROM brewflow_warehouse.fact_order_items;
-- Erwartet: 34 (33 aus Szenario B + 1 neue Late-arriving-Position)

-- Der entscheidende Check: welche Kundenversion wurde zugeordnet?
SELECT
    f.order_item_id,
    f.order_id,
    dc.customer_id,
    dc.loyalty_tier,
    dc.effective_from,
    dc.effective_to,
    dc.is_current
FROM brewflow_warehouse.fact_order_items f
JOIN brewflow_warehouse.dim_customer dc ON f.customer_sk = dc.customer_sk
WHERE f.order_id = 202;
-- Erwartet: loyalty_tier = Bronze, is_current = false
-- (NICHT Silver/true) -- das beweist den korrekten Point-in-Time-Join,
-- obwohl die Bestellung erst HEUTE geladen wurde, ihr order_date aber
-- vor dem Upgrade liegt.